# SwinJSCC — free GPU compatibility pilot

This notebook uses the pinned, unchanged author base-model release and an existing C=96 checkpoint. It checks strict checkpoint loading, native AWGN forward passes at training and full Kodak geometry, finite gradients, optimizer-state loading and checkpoint round-trip. Inputs are synthetic; it does **not** claim new real training epochs or Kodak quality results.

Use **Runtime → Change runtime type → T4 GPU**, then connect. Free GPU availability is not guaranteed. Do not select an upgrade or paid plan. The notebook does not purchase anything. The author source is checked by commit and file hashes before and after execution.

Upload the existing AWGN `model.pth` only when the upload cell asks. Original local weights are preserved. The optimizer probe writes a separately named synthetic checkpoint, which must never be counted as trained benchmark progress.

In [ ]:
from pathlib import Path
import sys, subprocess, json, hashlib, io, zipfile
import requests
import torch
assert torch.cuda.is_available(), 'No free GPU connected. Select a free T4 runtime; do not buy a plan.'
ROOT = Path('/content/SemanticLosslessPilot')
ROOT.mkdir(parents=True, exist_ok=True)
hardware = {'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0), 'gpu_count': torch.cuda.device_count()}
print(json.dumps(hardware, indent=2))
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
deps = ROOT / 'SharedCode/reproduction/.cpu_dependencies'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(deps), '--no-deps', 'timm==0.6.13', 'pytorch-msssim==1.0.0'], check=True)
print('Pilot dependencies installed; existing PyTorch is not replaced.')


In [ ]:
manifest = {'model': 'SwinJSCC', 'repository': 'https://github.com/semcomm/SwinJSCC', 'commit': 'a6d0e6da53548976acbe9317839a077ef31f190f', 'provenance': 'author_release', 'fetched_at': '2026-10-20T14:00:04.821434+00:00', 'archive_url': 'https://codeload.github.com/semcomm/SwinJSCC/zip/a6d0e6da53548976acbe9317839a077ef31f190f', 'archive_sha256': '9ec09e7e5205ac682246982882caed58bf988895b68866371944a9359bfeef9b', 'source_directory': 'Models/author_releases/SwinJSCC/a6d0e6da53548976acbe9317839a077ef31f190f', 'files_sha256': {'README.md': 'db4a77a985988c043730d4f0fb2dc2cc243ee069f6d05a53021442ca761513c4', 'data/datasets.py': '98aead75565c246a9d0cb997c03667088fc32ee8cdfa6baf57e3b9b7ca01db6f', 'loss/distortion.py': '8b02884a542b73db1c3a55faf698368132395aa2344feac0cc1c8083d8836402', 'main.py': '013f1bf13e7f4536173bdf061b602e723817a40776582206631d0b76cd3fd4f9', 'net/channel.py': '84572df9fb661d5e10cc0b00dbed1db473eef94bb1822833be06b33443dc74d9', 'net/decoder.py': '29260e7252551e445eae245c1714879e7459ca93a3dfeec9119da80136c87822', 'net/encoder.py': 'dd9379276d6cc2dbdd6ac174b14a72e604e859ed30884367e62f5a1e8bbb37d3', 'net/modules.py': '341213cd8f59e983045acd7354ca2d0768da58389fc346ee9516856ecd90e6ce', 'net/network.py': '215fc46a52809f9db57570bb8a1d1a278c936c30abd2018aa55efb7b0fc73b6a', 'utils.py': '48027e772236aca5ea62047d067546dd9bd4c2810e8add33b4d623b72ee72d0e'}, 'file_count': 10, 'execution_verified': False, 'paper_reproduction_verified': False}
source = ROOT / manifest['source_directory']
response = requests.get(manifest['archive_url'], timeout=(20,120))
response.raise_for_status()
with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    for item in archive.infolist():
        if item.is_dir(): continue
        relative = Path(*Path(item.filename).parts[1:])
        assert '..' not in relative.parts and not relative.is_absolute()
        destination = source / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(item))
for name, digest in manifest['files_sha256'].items():
    assert hashlib.sha256((source / name).read_bytes()).hexdigest() == digest, name
(source.parent / (manifest['commit'] + '.manifest.json')).write_text(json.dumps(manifest, indent=2))
print('Unchanged author source verified:', manifest['commit'])


In [ ]:
probe_source = '"""Real upstream Swin checkpoint/forward/backward/round-trip diagnostic.\n\nCPU mode directly exercises the unchanged encoder, decoder and distortion loss\nat the initialized training geometry. The network resolution updater and channel\nallocate CUDA tensors; neither is silently patched to run on CPU.\nGPU mode runs the original native AWGN channel. Synthetic input is used only for\ncompatibility checks; no benchmark quality or trained epochs are claimed.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport time\nfrom types import SimpleNamespace\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", type=Path, default=Path(__file__).resolve().parents[2])\n    parser.add_argument("--device", choices=["cpu", "cuda"], default="cpu")\n    parser.add_argument("--checkpoint", type=Path)\n    parser.add_argument("--output", type=Path)\n    args = parser.parse_args()\n    root = args.root.resolve()\n    out = args.output or root / f"Results/reproduction_audit/swin_{args.device}_runtime_probe"\n    out.mkdir(parents=True, exist_ok=True)\n    report = {"started_at": datetime.now(timezone.utc).isoformat(), "status": "RUNNING",\n              "device": args.device, "synthetic_inputs_only": True,\n              "benchmark_metrics_computed": False, "real_training_epochs_completed": 0,\n              "source_model_edits": [], "native_channel_executed": args.device == "cuda"}\n\n    def persist():\n        (out / "report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n\n    persist()\n    start = time.perf_counter()\n    try:\n        deps = root / "SharedCode/reproduction/.cpu_dependencies"\n        if deps.exists():\n            sys.path.insert(0, str(deps))\n        import torch\n        import torch.nn as nn\n        import timm\n        torch.set_num_threads(4)\n        torch.manual_seed(20261020)\n        if args.device == "cuda" and not torch.cuda.is_available():\n            raise RuntimeError("No CUDA device; GPU validation not performed")\n        report.update(torch_version=torch.__version__, timm_version=timm.__version__,\n                      cuda_available=torch.cuda.is_available())\n        manifests = list((root / "Models/author_releases/SwinJSCC").glob("*.manifest.json"))\n        if len(manifests) != 1:\n            raise RuntimeError("Exactly one pinned Swin release is required")\n        manifest = json.loads(manifests[0].read_text(encoding="utf-8"))\n        source = root / manifest["source_directory"]\n        for name, digest in manifest["files_sha256"].items():\n            if hashlib.sha256((source / name).read_bytes()).hexdigest() != digest:\n                raise RuntimeError("Changed upstream source: " + name)\n        sys.path.insert(0, str(source))\n        # Execute only the unchanged author configuration class. Importing the\n        # entire main script would also execute its CUDA-only metric setup.\n        tree = ast.parse((source / "main.py").read_text(encoding="utf-8"))\n        config_node = next(node for node in tree.body if isinstance(node, ast.ClassDef) and node.name == "config")\n        model_args = SimpleNamespace(trainset="DIV2K", testset="kodak", model_size="base",\n                                     model="SwinJSCC_w/o_SAandRA", C="96", multiple_snr="10",\n                                     channel_type="awgn", distortion_metric="MSE")\n        scope = {"torch": torch, "nn": nn, "datetime": datetime, "args": model_args}\n        exec(compile(ast.Module(body=[config_node], type_ignores=[]), str(source / "main.py"), "exec"), scope)\n        config = scope["config"]\n        config.device = torch.device(args.device)\n        config.CUDA = args.device == "cuda"\n        config.pass_channel = args.device == "cuda"\n        from net.network import SwinJSCC\n        model = SwinJSCC(model_args, config).to(args.device)\n        checkpoint_path = args.checkpoint or root / "Results/seven_model_results/SwinJSCC/AWGN/model.pth"\n        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n        state = checkpoint["model"]\n        prefix = "module.core."\n        if not all(key.startswith(prefix) for key in state):\n            raise RuntimeError("Unexpected checkpoint wrapper; refusing a partial/guessing load")\n        state = {key[len(prefix):]: value for key, value in state.items()}\n        incompatible = model.load_state_dict(state, strict=True)\n        report.update(source_commit=manifest["commit"], source_integrity="PASS",\n                      checkpoint_epoch=checkpoint["epoch"], checkpoint_strict_load="PASS",\n                      missing_keys=list(incompatible.missing_keys), unexpected_keys=list(incompatible.unexpected_keys),\n                      checkpoint_parameter_tensors=len(state), parameter_count=sum(p.numel() for p in model.parameters()),\n                      original_checkpoint_path=str(checkpoint_path))\n        persist()\n        print("STRICT CHECKPOINT LOAD PASS", flush=True)\n        def diagnostic_forward(x):\n            if args.device == "cuda":\n                return model(x)\n            feature = model.encoder(x, 10, 96, model_args.model)\n            rec = model.decoder(feature, 10, model_args.model)\n            mse = model.squared_difference(x * 255., rec.clamp(0., 1.) * 255.).mean()\n            loss = model.distortion_loss.forward(x, rec.clamp(0., 1.)).mean()\n            return rec, feature.numel() / 2 / x.numel(), 10, mse, loss\n\n        shapes = [(256, 256), (512, 768)] if args.device == "cuda" else [(256, 256)]\n        report["cpu_scope"] = "Direct unchanged encoder/decoder/loss at initialized 256x256; CUDA-only resolution updater and channel not tested" if args.device == "cpu" else None\n        report["full_kodak_geometry_checked"] = args.device == "cuda"\n        report["forward_checks"] = []\n        model.eval()\n        for height, width in shapes:\n            x = torch.rand(1, 3, height, width, device=args.device)\n            t0 = time.perf_counter()\n            with torch.inference_mode():\n                reconstructed, cbr, _, _, loss = diagnostic_forward(x)\n            if args.device == "cuda":\n                torch.cuda.synchronize()\n            if reconstructed.shape != x.shape or not torch.isfinite(reconstructed).all() or not torch.isfinite(loss):\n                raise RuntimeError("Invalid reconstruction shape or nonfinite output")\n            item = {"input_shape": list(x.shape), "output_shape": list(reconstructed.shape),\n                    "finite": True, "reported_cbr": float(cbr), "seconds": time.perf_counter()-t0}\n            report["forward_checks"].append(item)\n            persist()\n            print("FORWARD PASS", item, flush=True)\n            del x, reconstructed, loss\n        model.train()\n        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)\n        optimizer.load_state_dict(checkpoint["optimizer"])\n        optimizer.zero_grad(set_to_none=True)\n        x = torch.rand(1, 3, 256, 256, device=args.device)\n        t0 = time.perf_counter()\n        _, _, _, _, loss = diagnostic_forward(x)\n        if not torch.isfinite(loss):\n            raise RuntimeError("Nonfinite diagnostic training loss")\n        loss.backward()\n        gradients = [p.grad for p in model.parameters() if p.grad is not None]\n        if not gradients or not all(torch.isfinite(g).all() for g in gradients):\n            raise RuntimeError("Missing or nonfinite gradients")\n        optimizer.step()\n        probe = out / "synthetic_roundtrip.pt"\n        torch.save({"synthetic_probe_only": True, "production_epochs_completed": 0,\n                    "source_commit": manifest["commit"], "model": model.state_dict(),\n                    "optimizer": optimizer.state_dict()}, probe)\n        restored = torch.load(probe, map_location=args.device, weights_only=False)\n        if not all(torch.equal(value, restored["model"][key]) for key, value in model.state_dict().items()):\n            raise RuntimeError("Checkpoint round-trip differs")\n        optimizer.load_state_dict(restored["optimizer"])\n        report.update(backward_finite="PASS", optimizer_state_load="PASS", synthetic_optimizer_steps=1,\n                      diagnostic_step_seconds=time.perf_counter()-t0, checkpoint_roundtrip="PASS",\n                      source_checkpoint_overwritten=False, synthetic_checkpoint_bytes=probe.stat().st_size,\n                      status="PASS" if args.device == "cuda" else "PASS_CPU_CORE_ONLY", limitation="CPU mode directly exercises unchanged encoder/decoder/loss, without CUDA-only resolution update and channel. New PyTorch CPU runtime is not the native paper environment. This is compatibility evidence only.")\n        for name, digest in manifest["files_sha256"].items():\n            if hashlib.sha256((source / name).read_bytes()).hexdigest() != digest:\n                raise RuntimeError("Upstream source changed during validation")\n    except Exception as exc:\n        report.update(status="FAIL", error_type=type(exc).__name__, error=str(exc))\n        raise\n    finally:\n        report["elapsed_seconds"] = time.perf_counter()-start\n        persist()\n\n\nif __name__ == "__main__":\n    main()\n'
probe_path = ROOT / 'SharedCode/reproduction/runtime_probe_swin.py'
probe_path.parent.mkdir(parents=True, exist_ok=True)
probe_path.write_text(probe_source)


## Existing checkpoint

Upload the **AWGN C=96** checkpoint from the original SwinJSCC run. This diagnostic requires strict loading; it will fail on the wrong architecture rather than skipping weights.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly the AWGN model.pth checkpoint.'
CHECKPOINT = Path(next(iter(uploaded))).resolve()
assert CHECKPOINT.is_file()
del uploaded
print('Checkpoint file available:', CHECKPOINT.name, CHECKPOINT.stat().st_size, 'bytes')


In [ ]:
output = ROOT / 'Results/swin_gpu_runtime_probe'
result = subprocess.run([sys.executable, '-u', str(probe_path), '--root', str(ROOT), '--device', 'cuda', '--checkpoint', str(CHECKPOINT), '--output', str(output)], check=False)
report_path = output / 'report.json'
assert report_path.exists(), 'Probe did not produce a status report.'
report = json.loads(report_path.read_text())
print(json.dumps(report, indent=2))
assert result.returncode == 0 and report['status'] == 'PASS', 'GPU validation failed; inspect report before training.'


In [ ]:
# Retrieve actual diagnostic evidence even if later training is not started.
evidence = ROOT / 'swin_gpu_pilot_evidence.zip'
with zipfile.ZipFile(evidence, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(report_path, 'report.json')
    archive.writestr('hardware.json', json.dumps(hardware, indent=2))
    freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
    archive.writestr('environment.txt', freeze)
files.download(str(evidence))
print('Pilot completed. No DIV2K training epoch or Kodak metric was claimed.')
